# HomeValue — guided clinic and student workspace

This notebook has two modes:

1. **Guided clinic:** run the prepared cells, predict what will happen, inspect the evidence, and make a model decision.
2. **Exercise continuation:** extend the analysis and prepare the final submission after class.

[Read the complete Exercise 01 brief](https://github.com/lathrahul/modern-ai-ml-exercises/tree/main/exercise-01-homevalue) before continuing beyond the clinic. Sections 1–8 are the guided clinic; Section 9 begins the independent exercise continuation.

During the clinic, you are not expected to write the full modeling workflow from scratch. Your job is to connect the business question, model output, error pattern, and decision. Do not delete the AI-use disclosure.

## Clinic roadmap

By the end of the clinic, you should be able to:

- identify the row, target, prediction moment, and permitted inputs;
- compare a median baseline, simple regression, and multiple regression on the same validation homes;
- interpret a coefficient using units and conditional language;
- identify one residual or segment risk; and
- recommend **advance simple**, **advance multiple**, or **require more evidence**.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', 40)
RANDOM_STATE = 42

In [ ]:
# Use local files when the repository has been cloned. When this notebook is
# opened directly in Colab, load the same versioned files from GitHub.
local_data_dirs = [Path('../data'), Path('data'), Path('exercise-01-homevalue/data')]
data_dir = next((path for path in local_data_dirs if (path / 'train.csv').exists()), None)

if data_dir is not None:
    train_source = data_dir / 'train.csv'
    test_source = data_dir / 'test.csv'
    source_label = str(data_dir.resolve())
else:
    base_url = ('https://raw.githubusercontent.com/lathrahul/'
                'modern-ai-ml-exercises/main/exercise-01-homevalue/data')
    train_source = f'{base_url}/train.csv'
    test_source = f'{base_url}/test.csv'
    source_label = 'course repository on GitHub'

train = pd.read_csv(train_source)
test = pd.read_csv(test_source)
print(f'Loaded data from {source_label}')
print('Training data:', train.shape)
print('Test data:', test.shape)
train.head()

## 1. Frame the decision before modeling

Complete these statements before running a model:

- **One row:** one home observed before sale.
- **Target:** sale price.
- **Prediction moment:** before the home is sold.
- **Decision:** support an initial pricing conversation, not replace an appraisal.
- **Leakage check:** an input is invalid if it is only known after the sale.

**Your prediction:** Which permitted feature do you expect to be most useful, and why?

In [ ]:
target = 'sale_price'
simple_feature = 'living_area_sqft'
numeric_features = [
    'living_area_sqft', 'overall_quality', 'year_built', 'bedrooms',
    'full_bathrooms', 'garage_capacity', 'basement_sqft'
]
categorical_features = ['neighborhood', 'kitchen_quality', 'central_air']
model_features = numeric_features + categorical_features

audit = pd.DataFrame({
    'role': ['target', 'simple-model input', 'multiple-model inputs'],
    'columns': [target, simple_feature, ', '.join(model_features)]
})
audit

### Visual intuition: a fitted line is a prediction rule

Before fitting anything, look at a sample of homes. A line will assign one predicted price to each living-area value. The vertical gap between a point and the line will become its residual.

In [ ]:
sample = train.sample(350, random_state=RANDOM_STATE)
plt.figure(figsize=(8, 5))
plt.scatter(sample[simple_feature], sample[target], alpha=0.35, color='#1f7a5a')
plt.xlabel('Living area (square feet)')
plt.ylabel('Sale price ($)')
plt.title('Observed homes before fitting a line')
plt.grid(alpha=0.15)
plt.show()

## 2. Create one validation design and keep it fixed

The validation homes represent unseen cases. Every model below is compared on these same rows so that the comparison is fair.

In [ ]:
X = train[model_features].copy()
y = train[target].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print('Training homes:', len(X_train))
print('Validation homes:', len(X_valid))

## 3. Establish the median baseline

A model must improve on a credible simple rule. Here, every validation home receives the median sale price from the training data.

The notebook reports three complementary summaries:

- **MAE:** the typical absolute dollar miss;
- **RMSE:** a dollar error that gives extra weight to large misses; and
- **R²:** improvement relative to a mean-based benchmark, not a dollar error and not a causal claim.

Ordinary least squares fits coefficients using squared training residuals. That fitting objective is related to MSE and RMSE, but model selection below uses held-out validation evidence.

**Predict first:** Will a living-area model reduce the typical dollar miss by a little or a lot?

In [ ]:
def regression_metrics(actual, predicted):
    return {
        'MAE': mean_absolute_error(actual, predicted),
        'RMSE': mean_squared_error(actual, predicted) ** 0.5,
        'R2': r2_score(actual, predicted)
    }

median_price = y_train.median()
baseline_predictions = np.repeat(median_price, len(y_valid))
baseline_metrics = regression_metrics(y_valid, baseline_predictions)
pd.DataFrame([baseline_metrics], index=['Median baseline']).style.format({
    'MAE': '${:,.0f}', 'RMSE': '${:,.0f}', 'R2': '{:.3f}'
})

## 4. Fit a simple living-area model

The coefficient reports the predicted price change for one additional square foot. It describes an association in this fitted model; it does not prove that adding a square foot causes the price change.

In [ ]:
simple_model = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('model', LinearRegression())
])
simple_model.fit(X_train[[simple_feature]], y_train)
simple_predictions = simple_model.predict(X_valid[[simple_feature]])
simple_metrics = regression_metrics(y_valid, simple_predictions)

simple_coefficient = simple_model.named_steps['model'].coef_[0]
print(f'Living-area coefficient: ${simple_coefficient:,.0f} per additional square foot')
pd.DataFrame([simple_metrics], index=['Simple regression']).style.format({
    'MAE': '${:,.0f}', 'RMSE': '${:,.0f}', 'R2': '{:.3f}'
})

### Mini-interaction: change the living area

Change `example_area` and rerun the cell. Before running it, predict the direction and approximate size of the change.

In [ ]:
example_area = 1800  # Change this value and rerun.
example_home = pd.DataFrame({simple_feature: [example_area]})
example_prediction = simple_model.predict(example_home)[0]
print(f'Predicted price for {example_area:,} square feet: ${example_prediction:,.0f}')

## 5. Fit a leakage-safe multiple-regression pipeline

Numeric and categorical inputs require different preparation. The pipeline learns missing-value replacements and category encodings from the training homes, then applies the same rules to validation homes.

**Predict first:** Will the multiple model improve the average home, the expensive-home segment, both, or neither?

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])
multiple_model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', LinearRegression())
])
multiple_model.fit(X_train, y_train)
multiple_predictions = multiple_model.predict(X_valid)
multiple_metrics = regression_metrics(y_valid, multiple_predictions)
pd.DataFrame([multiple_metrics], index=['Multiple regression']).style.format({
    'MAE': '${:,.0f}', 'RMSE': '${:,.0f}', 'R2': '{:.3f}'
})

In [ ]:
comparison = pd.DataFrame([
    {'Model': 'Median baseline', **baseline_metrics},
    {'Model': 'Simple regression', **simple_metrics},
    {'Model': 'Multiple regression', **multiple_metrics}
]).set_index('Model')
comparison.style.format({
    'MAE': '${:,.0f}',
    'RMSE': '${:,.0f}',
    'R2': '{:.3f}'
})

## 6. Interpret selected coefficients carefully

Read each numeric coefficient as a comparison among homes with the other included features held fixed. Do not use the word *impact* or claim causality.

In [ ]:
feature_names = multiple_model.named_steps['preprocess'].get_feature_names_out()
coefficients = pd.Series(
    multiple_model.named_steps['model'].coef_, index=feature_names
).sort_values(key=np.abs, ascending=False)
selected_numeric = [f'num__{name}' for name in numeric_features]
coefficients.loc[selected_numeric].rename('coefficient').round(0).to_frame()

## 7. Diagnose residuals and segment risk

A metric summarizes error. A residual plot shows whether the errors have a pattern. A credible model should not systematically miss the same kinds of homes.

In [ ]:
residuals = y_valid.to_numpy() - multiple_predictions
plt.figure(figsize=(8, 5))
plt.scatter(multiple_predictions, residuals, alpha=0.45, color='#2563a9')
plt.axhline(0, color='#b0413e', linestyle='--', linewidth=2)
plt.xlabel('Predicted sale price ($)')
plt.ylabel('Residual: actual - predicted ($)')
plt.title('Multiple-model residuals')
plt.grid(alpha=0.15)
plt.show()

In [ ]:
high_price_cutoff = y_valid.quantile(0.80)
high_price_mask = y_valid >= high_price_cutoff
segment_results = pd.DataFrame({
    'Segment': ['All validation homes', 'Highest-price 20%'],
    'Homes': [len(y_valid), int(high_price_mask.sum())],
    'MAE': [
        mean_absolute_error(y_valid, multiple_predictions),
        mean_absolute_error(y_valid[high_price_mask], multiple_predictions[high_price_mask])
    ],
    'Mean residual': [
        residuals.mean(),
        residuals[high_price_mask.to_numpy()].mean()
    ]
})
segment_results.round(0)

## 8. Make the clinic decision

Choose **advance simple**, **advance multiple**, or **require more evidence**.

Your four-bullet coefficient memo must include:

1. one coefficient interpreted in business units;
2. one validation metric compared with the median baseline;
3. one residual or segment risk; and
4. one safe claim, unsafe claim, or next test.

Save the notebook. The polished Exercise 01 submission comes later.

## 9. Exercise continuation and final submission

After the clinic, revisit the full problem brief. You may revise the feature set and model, but preserve a leakage-safe split, compare against a baseline, and document why the final model is appropriate.

In [ ]:
# Fit the model you ultimately choose on all labeled training rows, then predict the test rows.
# Do not run this section until you have documented your model choice.
# final_model = ...
# final_model.fit(train[final_features], train[target])
# predictions = final_model.predict(test[final_features])
# submission = pd.DataFrame({
#     'property_id': test['property_id'],
#     'predicted_sale_price': predictions
# })
# submission.to_csv('../submission.csv', index=False)

## Generative AI use disclosure

Complete the disclosure specified in `../../shared/ai_use_disclosure.md`.